# UniSchema egress report

Reads **ConstituentEvent** JSON from local egress and summarizes donations by source system.

**Prerequisite:** run `npm run demo` from the repo root (or send a test webhook).

In [ ]:
import json
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "downstream" else Path.cwd()
EGRESS_DIR = REPO_ROOT / "data" / "egress"

rows = []
for path in EGRESS_DIR.rglob("*.json"):
    if "manifest" in path.name:
        continue
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict) and "eventId" in data:
        rows.append(data)

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} ConstituentEvent(s) from {EGRESS_DIR}")
df.head()

In [ ]:
if df.empty:
    print("No events yet — run: npm run demo")
else:
    summary = df.groupby(["sourceSystem", "eventType"], dropna=False).agg(
        events=("eventId", "count"),
        donation_total=("amount", "sum"),
    )
    display(summary)

    if "amount" in df.columns and df["amount"].notna().any():
        donations = df[df["eventType"] == "DONATION"]
        donations.groupby("sourceSystem")["amount"].sum().plot(kind="bar", title="Donations by source system")

## Feature table (PhilanthroPy bridge)

Aggregates events per `constituent_email`. See `docs/philanthropy-integration.md`.

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "examples" / "downstream"))

from unischema_features import egress_dir_to_features

if df.empty:
    print("No events — run: npm run demo:multi")
else:
    features = egress_dir_to_features(EGRESS_DIR)
    display(features.head())

## PhilanthroPy affinity scores (optional)

Requires: `pip install -r examples/downstream/requirements-philanthropy.txt`

In [ ]:
try:
    from philanthropy.models import DonorPropensityModel
    import numpy as np

    if not df.empty and len(features) >= 2:
        X = features[["total_gift_amount", "years_active", "event_attendance_count"]].to_numpy(dtype=float)
        proxy = (
            features["donation_count"] * 3
            + features["registration_count"] * 2
            + features["email_click_count"]
        )
        y = (proxy >= 4).astype(int).to_numpy()
        if len(set(y)) >= 2:
            model = DonorPropensityModel(n_estimators=50, random_state=42)
            model.fit(X, y)
            scores = model.predict_affinity_score(X)
            features.assign(affinity_score=scores)["affinity_score"].plot(
                kind="hist", title="PhilanthroPy affinity scores (demo proxy labels)"
            )
        else:
            print("Add more varied events for class diversity")
    else:
        print("Need more events — run: npm run demo:multi")
except ImportError:
    print("Install PhilanthroPy: pip install -r examples/downstream/requirements-philanthropy.txt")

## S3 NDJSON batches (production)

Download a batch from S3, then:

```python
lines = open("batch.ndjson").read().splitlines()
df = pd.DataFrame(json.loads(line) for line in lines if line.strip())
```

Or use `read_s3_ndjson_batch.py` from the shell.